In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import csv
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score,recall_score,f1_score,roc_auc_score
from sklearn.preprocessing import StandardScaler as standard_scaler




###Task 3.1 — Features, split and baseline

In [ ]:
# load the appointment data
appointments_clean=pd.read_csv("appointments_clean.csv")
#display headers
print(f"headers:",appointments_clean.columns)
df=appointments_clean
# map reminder_sent to 1 orr 0
df['reminder_sent'] = df['reminder_sent'].map({"Yes": 1, "No": 0})
# one-hot encoded booking_channel and visit_type
df=pd.get_dummies(df, columns=["booking_channel","visit_type"])
# print(df.head(5))
# setting X
X=df.drop(columns=['patient_id', 'appointment_id', 'booked_on','appt_date','clinic_id','gender','status','no_show'])
# setting Y
y=df['no_show']
# Split with train_test_split(X, y, test_size=0.2, random_state=42)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
# Report the training and test sizes
print(f"training size: {X_train.shape[0]}")
print(f"test size: {X_test.shape[0]}")
# no-shows in the test set
print(f"no-shows in the test set: {y_test.value_counts()[1]}")
# baseline accuracy of always predicting "Attended"
print(f"baseline accuracy: {round(y_test.value_counts()[0]/(y_test.value_counts()[0]+y_test.value_counts()[1]),4)}")



headers: Index(['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
       'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
       'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days',
       'no_show'],
      dtype='object')
training size: 792
test size: 198
no-shows in the test set: 49
baseline accuracy: 0.7525


### Task 3.2 — Logistic regression

In [ ]:
# Scale with StandardScaler
scaler=standard_scaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
# model selection and fitting
model_lr=LogisticRegression(max_iter=1000)
model_lr.fit(X_train_s,y_train)
y_pred=model_lr.predict(X_test_s)
y_prob=model_lr.predict_proba(X_test_s)[:,1]
# report accuracy, the confusion matrix, precision, recall, F1 and ROC-AUC
accuracy_lr=accuracy_score(y_test,y_pred)
precision_lr= precision_score(y_test,y_pred)
recall_lr= recall_score(y_test,y_pred)
f1_lr= f1_score(y_test,y_pred)
roc_auc_lr= roc_auc_score(y_test,y_prob)
confusion_matrix_lr= confusion_matrix(y_test,y_pred)
print(f"accuracy_lr: {round(accuracy_lr,4)}")
print(f"precision_lr: {round(precision_lr,4)}")
print(f"recall_lr: {round(recall_lr,4)}")
print(f"f1_lr: {round(f1_lr,4)}")
print(f"roc_auc_lr: {round(roc_auc_lr,4)}")
print(f"confusion_matrix_lr: {confusion_matrix_lr}")



accuracy_lr: 0.7879
precision_lr: 0.6296
recall_lr: 0.3469
f1_lr: 0.4474
roc_auc_lr: 0.7889
confusion_matrix_lr: [[139  10]
 [ 32  17]]


In [ ]:
# print features,model coef
coefficients=pd.DataFrame({"features":X.columns,"coef":model_lr.coef_[0]})
coefficients=coefficients.sort_values(by="coef",ascending=False)
print(coefficients)
#  report np.exp(coefficient) for reminder_sent
# Find the reminder_sent feature position
reminder_index = list(X.columns).index("reminder_sent")

# Extract the reminder coefficient
reminder_coef = model_lr.coef_[0][reminder_index]

# Odds multiplier for one standard deviation
odds_multiplier_scaled = np.exp(reminder_coef)

# Adjust coefficient to represent a full 0-to-1 change
reminder_coef_original = (
    reminder_coef / scaler.scale_[reminder_index]
)

# Odds multiplier for changing reminder from No to Yes
odds_multiplier_actual = np.exp(reminder_coef_original)

print("Reminder coefficient:", round(reminder_coef, 4))

print(
    "Odds multiplier per standard deviation:",
    round(odds_multiplier_scaled, 3)
)

print(
    "Odds multiplier for No to Yes:",
    round(odds_multiplier_actual, 3)
)
percentage_change=(1-odds_multiplier_actual)*100
print(f"percentage_change: {round(percentage_change,2)}%")

                   features      coef
4            prior_no_shows  0.649612
5                 lead_days  0.614471
2                   fee_inr  0.452761
1               distance_km  0.253448
7     booking_channel_Phone  0.215881
9   visit_type_Consultation  0.015658
10     visit_type_Follow-up -0.015658
6       booking_channel_App -0.046804
8   booking_channel_Walk-In -0.186364
0                       age -0.253880
3             reminder_sent -0.580669
Reminder coefficient: -0.5807
Odds multiplier per standard deviation: 0.56
Odds multiplier for No to Yes: 0.309
percentage_change: 69.06%


In this logistic regression model, holding the other features constant, appointments with reminders have approximately 69% lower odds of a no-show than appointments without reminders. This is an association in observational data and does not establish that reminders cause the reduction.

The two odds multipliers differ because the first represents a one-standard-deviation change in the scaled feature, whereas the second represents the full change from 0 (No) to 1 (Yes).

###**Task 3.3 — Decision tree and overfitting**

In [ ]:
# Full-depth decision tree
tree_full = DecisionTreeClassifier(random_state=42)

tree_full.fit(X_train, y_train)

# Depth-limited decision tree
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_depth3.fit(X_train, y_train)

print("Both decision tree models trained successfully!")
# Full-depth tree accuracy
full_train_accuracy = accuracy_score(
    y_train,
    tree_full.predict(X_train)
)

full_test_accuracy = accuracy_score(
    y_test,
    tree_full.predict(X_test)
)

# Depth-3 tree accuracy
depth3_train_accuracy = accuracy_score(
    y_train,
    tree_depth3.predict(X_train)
)

depth3_test_accuracy = accuracy_score(
    y_test,
    tree_depth3.predict(X_test))
# Display results
tree_results = pd.DataFrame({
    "Model": ["Full-depth Tree", "Depth-3 Tree"],
    "Training Accuracy": [
        full_train_accuracy,
        depth3_train_accuracy
    ],
    "Test Accuracy": [
        full_test_accuracy,
        depth3_test_accuracy
    ]
})

display(tree_results)

print("Full tree depth:", tree_full.get_depth())

Both decision tree models trained successfully!


,Model,Training Accuracy,Test Accuracy
0,Full-depth Tree,1.00000,0.676768
1,Depth-3 Tree,0.82197,0.777778


Full tree depth: 15


The full-depth decision tree achieves perfect training accuracy but significantly lower test accuracy, indicating overfitting because it memorises the training data rather than generalising well. Limiting the tree to depth 3 reduces overfitting and improves its test performance.

###Task 3.4 — Random forest and a comparison table

In [ ]:
# import randomforestclaasifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_rf=RandomForestClassifier(n_estimators=100,random_state=42)
model_rf.fit(X_train,y_train)
y_pred_rf=model_rf.predict(X_test)
y_proba_rf=model_rf.predict_proba(X_test)[:,1]
# Report accuracy, recall and ROC-AUC
accuracy_rf=accuracy_score(y_test,y_pred_rf)
precision_rf= precision_score(y_test,y_pred_rf)
recall_rf= recall_score(y_test,y_pred_rf)
roc_auc_rf= roc_auc_score(y_test,y_proba_rf)
random_forest_results=pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [accuracy_rf],
    "Precision": [precision_rf],
    "Recall": [recall_rf],
    "ROC-AUC": [roc_auc_rf]
})
print(random_forest_results)




           Model  Accuracy  Precision    Recall   ROC-AUC
0  Random Forest  0.747475   0.482759  0.285714  0.702301


In [ ]:

# Extract feature importance values
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model_rf.feature_importances_
})

# Sort from highest to lowest
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("TOP THREE IMPORTANT FEATURES")

display(feature_importance.head(3))


TOP THREE IMPORTANT FEATURES


,Feature,Importance
5,lead_days,0.222765
0,age,0.202164
1,distance_km,0.192075


Feature importance measures how much each feature contributes to the random forest's decision-making. Lead time, age and distance are among the most influential features in the reference results.

In [ ]:
# build a table with one row each for logistic regression, the depth-3 tree
#  and the random forest (accuracy, recall, ROC-AUC)
comparision_table=pd.DataFrame({
    "Model": ["Logistic Regression", "Depth-3 Tree", "Random Forest"],
    "Accuracy": [accuracy_lr,depth3_test_accuracy,accuracy_rf],
    "recall": [recall_lr, recall_score(y_test, tree_depth3.predict(X_test)),recall_rf],
    "ROC-AUC": [
        roc_auc_lr,
        roc_auc_score(
            y_test,
            tree_depth3.predict_proba(X_test)[:, 1]
        ),
        roc_auc_rf
    ]})
print(comparision_table)


                 Model  Accuracy    recall   ROC-AUC
0  Logistic Regression  0.787879  0.346939  0.788933
1         Depth-3 Tree  0.777778  0.224490  0.699699
2        Random Forest  0.747475  0.285714  0.702301


Logistic Regression is the best-performing model among the three evaluated models. It achieved the highest accuracy (78.79%), recall (34.69%), and ROC-AUC (0.7889) on the test dataset. In particular, its higher recall means it identifies more actual no-show appointments than the Decision Tree and Random Forest models. Therefore, Logistic Regression is selected as the preferred model for further hyperparameter evaluation and threshold optimization.

###Task 3.5 — Tune, choose the model and the threshold, then test once

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Logistic Regression pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Depth-3 Decision Tree
depth3_tree = DecisionTreeClassifier(max_depth=3, random_state=42)

# Default Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation ROC-AUC:", grid_search.best_score_)

tuned_rf = grid_search.best_estimator_
print(tuned_rf)

Best parameters: {'max_depth': 5, 'n_estimators': 100}
Best cross-validation ROC-AUC: 0.773589615318268
RandomForestClassifier(max_depth=5, random_state=42)


(b) Choose the final model without touching the test set: compute the 5-fold cross-validated ROC-AUC on the training set for logistic regression, the depth-3 tree and the random forest — cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc").mean() — and compare them with the tuned forest's best_score_. For logistic regression, give cross_val_score the unscaled X_train together with the supplied pipeline below, so that the scaler is re-fitted on each fold's own training part (scaling the whole training set first would let each validation fold influence the scaler). The highest wins.



In [ ]:
lr_cv = cross_val_score(
    lr_pipeline, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tree_cv = cross_val_score(
    depth3_tree, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

rf_cv = cross_val_score(
    rf_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv = grid_search.best_score_

cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Depth-3 Tree",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "CV ROC-AUC": [
        lr_cv,
        tree_cv,
        rf_cv,
        tuned_rf_cv
    ]
})

print(cv_results.sort_values("CV ROC-AUC", ascending=False))

# Select the highest-scoring model
models = {
    "Logistic Regression": (lr_pipeline, lr_cv),
    "Depth-3 Tree": (depth3_tree, tree_cv),
    "Random Forest": (rf_model, rf_cv),
    "Tuned Random Forest": (tuned_rf, tuned_rf_cv)
}

best_name = None
best_score = float("-inf")

for name, details in models.items():
    score = details[1]
    if score > best_score:
        best_score = score
        best_name = name

print("\nSelected model:", best_name)
print("Selected CV ROC-AUC:", round(models[best_name][1],2))

                 Model  CV ROC-AUC
0  Logistic Regression    0.773714
3  Tuned Random Forest    0.773590
2        Random Forest    0.741987
1         Depth-3 Tree    0.678449

Selected model: Logistic Regression
Selected CV ROC-AUC: 0.77


The model with the highest mean cross-validated ROC-AUC wins. Do not select based on the earlier test-set comparison.

####
(c) Choose the threshold using out-of-fold training probabilities

In [ ]:
# Generate out-of-fold probabilities using training data only
oof_prob = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Compare thresholds
threshold_results = []

for threshold in [0.50, 0.35]:
    predictions = (oof_prob >= threshold).astype(int)


    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, predictions),
        "Recall": recall_score(y_train, predictions),
        "F1": f1_score(y_train, predictions),
        "Number Flagged": predictions.sum()
    })

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df)

# Select threshold with the higher F1
best_threshold = threshold_df.loc[
    threshold_df["F1"].idxmax(), "Threshold"
]

print("\nSelected threshold:", best_threshold)

   Threshold  Precision    Recall        F1  Number Flagged
0       0.50   0.575342  0.248521  0.347107              73
1       0.35   0.512987  0.467456  0.489164             154

Selected threshold: 0.35


####(d) Fit the winner and evaluate once on the test set

In [ ]:
# Fit selected model on training data
best_model.fit(X_train, y_train)

# Predict test probabilities
test_prob = best_model.predict_proba(X_test)[:, 1]

# Apply selected threshold
test_pred = (test_prob >= best_threshold).astype(int)

# Evaluate test performance
print("Selected model:", best_name)
print("Selected threshold:", best_threshold)
print("Test precision:", precision_score(y_test, test_pred))
print("Test recall:", recall_score(y_test, test_pred))
print("Number flagged:", test_pred.sum())
print("Test ROC-AUC:", roc_auc_score(y_test, test_prob))

Selected model: Logistic Regression
Selected threshold: 0.35
Test precision: 0.5714285714285714
Test recall: 0.5714285714285714
Number flagged: 49
Test ROC-AUC: 0.788933022873579


####(e)Export test_risk_scores.csv

In [ ]:
# Retrieve appointment IDs using original row indices
test_ids = appointments_clean.loc[X_test.index, "appointment_id"]

# Create risk score DataFrame
risk_scores = pd.DataFrame({
    "appointment_id": test_ids.values,
    "no_show": y_test.values,
    "risk_prob": test_prob,
    "flag": test_pred
})

# Export CSV
risk_scores.to_csv("test_risk_scores.csv", index=False)

print(risk_scores.head())
print("\nRows:", len(risk_scores))
print("Columns:", risk_scores.columns.tolist())

  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0

Rows: 198
Columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']


Final Model Selection and Threshold Optimization

The four candidate models were compared using 5-fold cross-validation ROC-AUC on the training dataset. Logistic regression model achieved the highest cross-validated ROC-AUC of 0.77 and was selected as the final model.

Out-of-fold training predictions were used to compare classification thresholds of 0.50 and 0.35. A threshold of 0.35 was selected because it achieved the higher F1-score of 0.489, while providing a useful balance between precision and recall.

The selected model was then fitted on the full training dataset and evaluated once on the test dataset. It achieved a precision of 0.57, recall of 0.571, ROC-AUC of 0.788, and flagged 49 appointments as high risk.

In practice, a clinic would choose its threshold based on how many follow-up calls the front desk can make in an evening, balanced against the cost of leaving appointment slots empty.